# Lab: k-Means from the Ground Up

**Bluevision-ai Academy**
*k-Means: Finding Groups Without Labels*

In lecture we watched the answer column vanish, picked an objective (nearest prototype), built the assign/update loop, watched it converge, and then watched it break. This lab makes every one of those claims yourself, in code, on data you can inspect.

Each part runs the same rhythm the lecture did, just in cells instead of clicks: **Problem** (what we have, what we want, why the obvious tool doesn't reach it) -> **Idea** (the reasoning move, in words, before any code) -> **Build** (the code that embodies exactly that idea) -> **Payoff** (what just happened, and what it means). One idea per cell -- if a cell needs "and" to describe it, it's doing too much.

| Part | What you build | Ties back to |
|---|---|---|
| 1 -- From Scratch | The assign/update loop on real flower measurements; watch WCSS fall | Slides 6-8 |
| 2 -- Initialisation | Two engineered seed sets, one good one bad; then k-means++ | Slides 9-10 |
| 3 -- Choosing k | Elbow + silhouette on the same data; peek at the real answer once | Slide 11 |
| Challenge | Break it on purpose: elongated clusters | Slide 12 |
| Reflection | Five questions, answered in code comments | -- |

**Dataset:** the Iris flower measurements, arguably the most famous dataset in all of machine learning, and a good fit here: three real species, no label column shown to the algorithm until the very end. **Estimated time:** ~2 hours. Runs top to bottom on Google Colab with no setup beyond the cell below.

## Setup

Colab already ships with everything this lab needs. The cell below just makes sure versions are current, turns on interactive widgets, and loads the deck's own palette so these plots read like the ones you just watched.

In [1]:
# Quiet, idempotent installs -- safe to re-run.
!pip install -q --upgrade scikit-learn plotly ipywidgets

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider, fixed

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # not running on Colab -- local Jupyter with ipywidgets already works

np.random.seed(0)

# The deck's own palette, so the plots here read like the plots you just watched.
GOLD, BLUE, TEAL, RED, DIM = "#b07d22", "#2563a8", "#148a7a", "#c0392b", "#7f8c9a"

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## Part 1 -- k-Means from scratch

**Problem.** Slide 4 left us with a cloud of points and no ground truth: any grouping is only ever a choice of objective, never a fact hiding in the data. Slide 6 picked the simplest workable objective anyone has come up with: represent each group by a single prototype, and let the nearest prototype claim every point. We have real flower measurements below and, per that objective, we want an algorithm that finds good prototypes entirely on its own, starting from nothing.

**Ingredients.** The Iris dataset: 150 flowers, four measurements each. We deliberately keep only two of them, petal length and petal width, so every plot below is the real geometry, not a projection you have to trust. The species column exists in the data, but we do not load it here -- the villain from slide 2, this time for real.

In [2]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data[:, [2, 3]]  # petal length, petal width only -- plots straight in 2D
feature_names = ["petal length (cm)", "petal width (cm)"]

fig = go.Figure(go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers",
                            marker=dict(color=DIM, size=7)))
fig.update_layout(title="150 flowers, two measurements each -- no species column loaded",
                   xaxis_title=feature_names[0], yaxis_title=feature_names[1],
                   width=650, height=450)
fig.show()

**Idea -- assign, update, repeat.** Start with k guessed prototypes. **Assign** every point to whichever prototype is nearest. **Update** every prototype to the mean of the points now assigned to it. Neither step alone finds anything -- but alternating them, slide 7's centrepiece animation, walks the prototypes into the data until nothing moves. That is the whole algorithm; there is no hidden machinery beneath it.

In [3]:
def kmeans_from_scratch(X, k, n_iter=10, seed=0):
    """Returns a history dict so we can replay every step afterwards,
    not just inspect the final answer."""
    rng = np.random.default_rng(seed)
    centroids = X[rng.choice(len(X), size=k, replace=False)]
    history = {"assign": [], "centroids": [centroids.copy()], "inertia": []}

    for _ in range(n_iter):
        # Assign: nearest centroid for every point.
        dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
        assign = np.argmin(dists, axis=1)

        # Score the current partition before moving anything.
        inertia = sum(
            np.sum((X[assign == j] - centroids[j]) ** 2) for j in range(k)
        )
        history["assign"].append(assign.copy())
        history["inertia"].append(float(inertia))

        # Update: each centroid becomes the mean of its own points.
        new_centroids = np.array([
            X[assign == j].mean(axis=0) if np.any(assign == j) else centroids[j]
            for j in range(k)
        ])
        centroids = new_centroids
        history["centroids"].append(centroids.copy())

    return history

**Idea -- trust, but verify.** Slide 8's claim was that this loop's objective, within-cluster sum of squares (WCSS, what scikit-learn calls inertia), only ever falls or holds steady, never rises. Run the loop on the flower data with k=3 and plot inertia after every iteration to check that claim against real numbers, not just take the slide's word for it.

In [4]:
hist_iris = kmeans_from_scratch(X, k=3, n_iter=8, seed=1)

fig = go.Figure(go.Scatter(y=hist_iris["inertia"], mode="lines+markers", line=dict(color=GOLD)))
fig.update_layout(title="Inertia (WCSS) by iteration -- petal measurements, k=3",
                   xaxis_title="iteration", yaxis_title="inertia", width=650, height=400)
fig.show()

assert hist_iris["inertia"][0] >= hist_iris["inertia"][-1], "inertia should not increase"

**Payoff.** The curve only drops -- the assert above would fail loudly if it didn't -- confirming slide 8's guarantee on real flowers, not just an abstract argument. It also flattens fast: almost all of the improvement happens in the first two or three iterations (686 to 142 to 39), then almost nothing changes. Keep that "flattens, doesn't warn" behaviour in mind; it comes back in the Challenge below.

**Idea -- watch the two steps trade off, one at a time.** The inertia curve hides the mechanism: which points moved, and where each prototype travelled to get there. Step through the same run one iteration at a time.

In [5]:
def plot_kmeans_state(X, history, t):
    assign, cent = history["assign"][t], history["centroids"][t]
    colors = [GOLD, BLUE, TEAL]
    fig = go.Figure()
    for j in range(cent.shape[0]):
        pts = X[assign == j]
        fig.add_trace(go.Scatter(x=pts[:, 0], y=pts[:, 1], mode="markers",
                                  marker=dict(color=colors[j % 3], size=7)))
        fig.add_trace(go.Scatter(x=[cent[j, 0]], y=[cent[j, 1]], mode="markers",
                                  marker=dict(color=colors[j % 3], size=18, symbol="star",
                                              line=dict(color="black", width=1))))
    fig.update_layout(title=f"iteration {t}  ·  inertia = {history['inertia'][t]:.1f}",
                       xaxis_title=feature_names[0], yaxis_title=feature_names[1],
                       width=650, height=500, showlegend=False)
    fig.show()

interact(plot_kmeans_state, X=fixed(X), history=fixed(hist_iris),
         t=IntSlider(min=0, max=len(hist_iris["assign"]) - 1, step=1, value=0));

interactive(children=(IntSlider(value=0, description='t', max=7), Output()), _dom_classes=('widget-interact',)…

**Payoff.** Drag the slider: colours settle and the stars stop moving within the first few steps, the same flattening the inertia curve already showed you, now with prototypes you can watch travel. Nobody told this algorithm there are three species of flower in this data. Hold that thought -- Part 3 comes back to it.

## Part 2 -- Initialisation: same data, different seeds

**Problem.** Slide 9 raised a question Part 1 quietly dodged: those first three prototypes in `kmeans_from_scratch` are drawn at random. Does *where* they start actually change *where* the algorithm ends up? Real Iris measurements turn out to be almost too well-behaved to show this cleanly -- the three species barely leave room for a bad answer. So for this part only, we build a small dataset on purpose: two natural groups sitting close together, and one sitting far off on its own, the exact shape that can trap a careless choice of seeds.

**Ingredients.** Thirty-six points in three round groups, twelve each: two groups near each other in the lower-left, one group well separated up top.

In [6]:
from sklearn.datasets import make_blobs

X_init, y_init = make_blobs(n_samples=36, centers=[(2, 2), (5, 2.6), (3.3, 5.8)],
                             cluster_std=0.8, random_state=3)

fig = go.Figure(go.Scatter(x=X_init[:, 0], y=X_init[:, 1], mode="markers",
                            marker=dict(color=DIM, size=8)))
fig.update_layout(title="Thirty-six points, three groups -- no colours yet", width=650, height=450)
fig.show()

**Idea -- two seed sets, on purpose.** Seed set A drops one prototype near each of the three groups -- exactly what slide 9 called the sensible start. Seed set B drops two prototypes into the same far-off group and only one prototype to cover both close-together groups. Same data, same algorithm, only the seeds differ.

In [7]:
def kmeans_from_seeds(X, seed_centroids, n_iter=25):
    """Same assign/update loop as Part 1, but starting from an explicit
    centroid array instead of a random draw -- so we can compare seed
    choices directly."""
    centroids = np.array(seed_centroids, dtype=float)
    k = len(centroids)
    for _ in range(n_iter):
        dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
        assign = np.argmin(dists, axis=1)
        centroids = np.array([
            X[assign == j].mean(axis=0) if np.any(assign == j) else centroids[j]
            for j in range(k)
        ])
    inertia = sum(np.sum((X[assign == j] - centroids[j]) ** 2) for j in range(k))
    return assign, centroids, inertia

seeds_good = [(2, 2), (5, 2.6), (3.3, 5.8)]
seeds_bad = [(3.3, 5.8), (2.8, 5.2), (2, 2)]

assign_good, cent_good, inertia_good = kmeans_from_seeds(X_init, seeds_good)
assign_bad, cent_bad, inertia_bad = kmeans_from_seeds(X_init, seeds_bad)

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f"seed set A -- inertia {inertia_good:.1f}", f"seed set B -- inertia {inertia_bad:.1f}"))
for j, color in enumerate([GOLD, BLUE, TEAL]):
    pts_g = X_init[assign_good == j]
    fig.add_trace(go.Scatter(x=pts_g[:, 0], y=pts_g[:, 1], mode="markers",
                              marker=dict(color=color, size=7)), row=1, col=1)
    pts_b = X_init[assign_bad == j]
    fig.add_trace(go.Scatter(x=pts_b[:, 0], y=pts_b[:, 1], mode="markers",
                              marker=dict(color=color, size=7)), row=1, col=2)
fig.update_layout(width=950, height=450, showlegend=False)
fig.show()

print(f"seed set A cluster sizes: {[int((assign_good == j).sum()) for j in range(3)]}")
print(f"seed set B cluster sizes: {[int((assign_bad == j).sum()) for j in range(3)]}")

seed set A cluster sizes: [11, 13, 12]
seed set B cluster sizes: [9, 5, 22]


**Payoff.** Seed set A lands close to the true group sizes (12/12/12) at the lower of the two inertia scores. Seed set B settles somewhere clearly worse: the two prototypes dropped into the far-off group tear it into a 9-and-5 split, while the two close-together groups get folded into one 22-point cluster. Same data, same algorithm, only the starting seeds moved -- and the final answer is not the same clusters with the colours swapped, it is a genuinely different, worse partition. That is what slide 9 meant by a "local optimum": good enough for the seeds it started with, never guaranteed to be the best possible grouping.

**Idea -- spread the seeds on purpose.** Slide 10's fix: pick the first seed uniformly at random, then weight every seed after that toward whatever is farthest from the seeds already chosen. Implement that weighting and check, over many restarts, whether it actually lands in a bad partition -- inertia above 60, roughly midway between seed set A's 48.5 and seed set B's 76.4 -- less often than blind random seeding does.

In [8]:
def kmeans_pp_seed(X, k, rng):
    idx0 = rng.integers(len(X))
    centroids = [X[idx0]]
    for _ in range(k - 1):
        dist_sq = np.min([np.sum((X - c) ** 2, axis=1) for c in centroids], axis=0)
        probs = dist_sq / dist_sq.sum()
        idx = rng.choice(len(X), p=probs)
        centroids.append(X[idx])
    return np.array(centroids, dtype=float)

n_trials = 300
bad_random, bad_pp = 0, 0
for trial in range(n_trials):
    rng = np.random.default_rng(trial)
    random_seeds = X_init[rng.choice(len(X_init), size=3, replace=False)]
    _, _, inertia_r = kmeans_from_seeds(X_init, random_seeds)
    if inertia_r > 60:
        bad_random += 1

    rng = np.random.default_rng(1000 + trial)
    pp_seeds = kmeans_pp_seed(X_init, 3, rng)
    _, _, inertia_p = kmeans_from_seeds(X_init, pp_seeds)
    if inertia_p > 60:
        bad_pp += 1

print(f"landed in a bad partition: random seeding {bad_random}/{n_trials}, "
      f"k-means++ seeding {bad_pp}/{n_trials}")

landed in a bad partition: random seeding 42/300, k-means++ seeding 29/300


**Payoff.** Random seeding lands in a bad partition on 42 of 300 restarts; k-means++ seeding on 29 -- roughly a third fewer, from one change: weighting new seeds toward whatever is farthest from the seeds already picked. Not zero, and not a guarantee -- an unlucky draw can still happen, and does, here -- but a real, measurable improvement, which is exactly why slide 10 called it the fix nearly every real implementation uses rather than a cure.

## Part 3 -- Choosing k: elbow and silhouette

**Problem.** Slide 11 raised a question nobody has answered yet: how many groups, exactly? Nobody handed `kmeans_from_scratch` a k in Parts 1 or 2 without us guessing it first. We want two independent ways to ask the data itself, and we want to know what to do when they don't agree.

**Ingredients.** The same hidden-species flower measurements from Part 1 -- `X`, already loaded, still no species column in sight.

**Idea -- the elbow.** Run k-means for a whole range of k and plot WCSS against k. More prototypes can only ever help the training objective, so the curve always falls -- what we are looking for is the k where it stops falling *fast*.

In [9]:
k_values = range(1, 9)
inertia_by_k = []
for k in k_values:
    h = kmeans_from_scratch(X, k, n_iter=15, seed=1)
    inertia_by_k.append(h["inertia"][-1])

fig = go.Figure(go.Scatter(x=list(k_values), y=inertia_by_k, mode="lines+markers", line=dict(color=GOLD)))
fig.update_layout(title="Elbow: WCSS vs k -- petal measurements",
                   xaxis_title="k", yaxis_title="inertia (WCSS)", width=650, height=400)
fig.show()

**Payoff.** WCSS keeps falling as k grows, exactly as promised. The drop from k=1 to k=2 is huge (551 to 86), and the drop from k=2 to k=3 is still substantial (86 to 31); past k=4 the curve is nearly flat. Read by eye, the "bend" could honestly be argued as either k=2 or k=3 -- the elbow alone does not cleanly pick one winner here.

**Idea -- the silhouette score.** For every point, compare how close it sits to its own cluster against how close it sits to the nearest other cluster, and average that score across every point. A point deep inside its group scores near one; a point sitting right on a boundary scores near zero. Unlike WCSS, this score does not automatically favour more clusters, so its peak is a real vote for a specific k.

In [10]:
from sklearn.metrics import silhouette_score

k_values_sil = range(2, 9)
silhouette_by_k = []
for k in k_values_sil:
    h = kmeans_from_scratch(X, k, n_iter=15, seed=1)
    silhouette_by_k.append(silhouette_score(X, h["assign"][-1]))

best_k = list(k_values_sil)[int(np.argmax(silhouette_by_k))]

fig = go.Figure(go.Scatter(x=list(k_values_sil), y=silhouette_by_k, mode="lines+markers", line=dict(color=BLUE)))
fig.add_vline(x=best_k, line_dash="dash", line_color=RED)
fig.update_layout(title=f"Silhouette score vs k -- peak at k={best_k}",
                   xaxis_title="k", yaxis_title="silhouette (higher is better)", width=650, height=400)
fig.show()

**Payoff.** Silhouette peaks cleanly at k=2 (0.765), clearly above k=3 (0.66) -- a real vote, not a coin flip, and it disagrees with the elbow's more ambiguous reading. Per slide 11: in practice these two tools don't always agree, and when they don't, that disagreement is telling you something too.

**Idea -- see every k, not just the two the curves argued about.** A curve is an average over every point at once. Drag k and watch the partition itself change shape, one k at a time.

In [11]:
def plot_k_partition(k):
    h = kmeans_from_scratch(X, k, n_iter=15, seed=1)
    assign = h["assign"][-1]
    palette = [GOLD, BLUE, TEAL, RED, DIM, "#6a3d9a", "#8c8c00", "#00838f"]
    fig = go.Figure()
    for j in range(k):
        pts = X[assign == j]
        fig.add_trace(go.Scatter(x=pts[:, 0], y=pts[:, 1], mode="markers",
                                  marker=dict(color=palette[j % len(palette)], size=7)))
    fig.update_layout(title=f"k={k}  ·  inertia={h['inertia'][-1]:.1f}",
                       xaxis_title=feature_names[0], yaxis_title=feature_names[1],
                       width=650, height=500, showlegend=False)
    fig.show()

interact(plot_k_partition, k=IntSlider(min=1, max=8, step=1, value=3));

interactive(children=(IntSlider(value=3, description='k', max=8, min=1), Output()), _dom_classes=('widget-inte…

**Payoff.** At k=2 a clean, confident split appears; at k=3 that larger piece cracks into two touching sub-groups; past k=4 the extra prototypes start slicing through what looks, by eye, like a single group. Nothing about that judgment is mechanical -- it is the same tension the elbow and silhouette curves were arguing about a moment ago, just something you can now feel by dragging instead of reading off two numbers.

**Idea -- check the disagreement against reality, once.** Everything above assumed no ground truth, which is the honest default. This dataset happens to keep one anyway: the species each flower actually belongs to, deliberately never loaded until now. Look, once, and see which of k=2 or k=3 the real answer sides with.

In [12]:
from sklearn.metrics import adjusted_rand_score

species_names = iris.target_names   # not used during clustering -- inspected now, for the first time
y = iris.target
species_palette = [GOLD, BLUE, TEAL]  # setosa, versicolor, virginica

def majority_color(assign, y, palette):
    colors_by_cluster = {}
    for j in np.unique(assign):
        majority_species = np.bincount(y[assign == j]).argmax()
        colors_by_cluster[j] = palette[majority_species]
    return [colors_by_cluster[a] for a in assign]

h2 = kmeans_from_scratch(X, k=2, n_iter=15, seed=1)
h3 = kmeans_from_scratch(X, k=3, n_iter=15, seed=1)
assign2, assign3 = h2["assign"][-1], h3["assign"][-1]

fig = make_subplots(rows=1, cols=3, subplot_titles=(
    "true species (never used above)", "k=2 clusters", "k=3 clusters"))
fig.add_trace(go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers",
                          marker=dict(color=[species_palette[s] for s in y], size=6)), row=1, col=1)
fig.add_trace(go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers",
                          marker=dict(color=majority_color(assign2, y, species_palette), size=6)), row=1, col=2)
fig.add_trace(go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers",
                          marker=dict(color=majority_color(assign3, y, species_palette), size=6)), row=1, col=3)
fig.update_layout(width=1000, height=380, showlegend=False)
fig.show()

print(f"agreement with real species -- k=2: ARI={adjusted_rand_score(y, assign2):.3f}, "
      f"k=3: ARI={adjusted_rand_score(y, assign3):.3f}")

agreement with real species -- k=2: ARI=0.558, k=3: ARI=0.851


**Payoff.** The left panel is the answer nobody showed the algorithm. At k=3, clustering nearly reproduces it (ARI 0.851): setosa comes out perfectly, and only a handful of versicolor/virginica flowers land with the wrong neighbour -- which is exactly why silhouette dropped a little from k=2's score, those two species genuinely overlap in petal space. At k=2, versicolor and virginica get folded into one broad, visually clean cluster (ARI only 0.558) -- clean is not the same as correct. This is slide 11's warning made concrete: two independent tools disagreeing was itself information, not a bug to explain away.

## Challenge -- break it deliberately: elongated clusters

**Problem.** Slide 12 promised k-means has failure modes: it *assumes* round, similarly sized, similarly dense groups. Every dataset in Parts 1 to 3 has been comfortably round-ish. Will the identical code hold up outside its comfort zone?

**Ingredients.** Three genuinely elongated clusters instead of round ones -- same total point count, same k, the same `kmeans_from_scratch` function, no code changes at all.

In [13]:
def make_elongated_cluster(center, angle_deg, n, stretch=3.0, base_std=0.35, rng=None):
    theta = np.deg2rad(angle_deg)
    R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    pts = rng.normal(0, 1, size=(n, 2)) * np.array([stretch, 1.0]) * base_std
    return pts @ R.T + np.array(center)

rng = np.random.default_rng(13)
X_elong = np.vstack([
    make_elongated_cluster((1.5, 3), 30, 30, rng=rng),
    make_elongated_cluster((5, 3.3), -25, 30, rng=rng),
    make_elongated_cluster((3.7, 6.8), 80, 30, rng=rng),
])

fig = go.Figure(go.Scatter(x=X_elong[:, 0], y=X_elong[:, 1], mode="markers",
                            marker=dict(color=DIM, size=7)))
fig.update_layout(title="Ninety points, three elongated clusters -- no colours yet",
                   width=650, height=450)
fig.show()

**Idea.** Run the identical `kmeans_from_scratch` on `X_elong` and look at both the final partition and its inertia curve -- not just one or the other.

In [14]:
hist_elong = kmeans_from_scratch(X_elong, k=3, n_iter=8, seed=1)
final_assign = hist_elong["assign"][-1]

fig = go.Figure()
for j, color in enumerate([GOLD, BLUE, TEAL]):
    pts = X_elong[final_assign == j]
    fig.add_trace(go.Scatter(x=pts[:, 0], y=pts[:, 1], mode="markers",
                              marker=dict(color=color, size=7)))
fig.update_layout(title="k-means's final cut on the elongated data",
                   width=650, height=500, showlegend=False)
fig.show()

print("inertia by iteration:", [round(v, 1) for v in hist_elong["inertia"]])

inertia by iteration: [264.2, 135.6, 127.6, 123.4, 118.7, 116.9, 116.9, 116.9]


**Payoff -- your answer (2-3 sentences).** Look at the plot above: where exactly does k-means cut, and why is that cut in the wrong place for the true elongated shapes? Name the assumption about cluster *shape* that just broke. Then look at the inertia sequence you just printed -- did it warn you anything had gone wrong?

*(write your answer here)*

## Reflection

**Problem.** You now own the full k-means toolkit: the algorithm, why it converges, how to seed it well, how to choose k, and exactly where it breaks. The point of this last cell isn't a right answer -- it's stating, in your own words, the reasoning each part was built on.

In [15]:
# 1. Part 1's inertia curve fell smoothly and flattened fast on the iris
#    flowers. Part 2's engineered "bad" seed set produced a clearly worse
#    partition than the "good" one, at a HIGHER final inertia. Put these two
#    observations together: can inertia, on its own, tell you whether a
#    k-means run found a GOOD partition, or only that it found A partition
#    it can no longer improve?
answer_1 = """
"""

# 2. In Part 2, k-means++ seeding landed in a "bad" partition on roughly a
#    third fewer of 300 restarts than blind random seeding did -- but not
#    zero. What does "roughly a third fewer, not zero" tell you about what
#    k-means++ actually guarantees?
answer_2 = """
"""

# 3. Part 3's elbow curve and silhouette score disagreed about whether k=2
#    or k=3 was the better choice on the same flower data. Once you saw the
#    real species names, explain in your own words WHY they disagreed --
#    what property of the data causes exactly this kind of disagreement?
answer_3 = """
"""

# 4. The Challenge ran the identical `kmeans_from_scratch` function on
#    elongated clusters with zero code changes. Name the one assumption
#    about cluster SHAPE that broke, and describe what a fix would have to
#    let a cluster's boundary do that a single distance number cannot.
answer_4 = """
"""

# 5. You now have four tools from this lab: the raw algorithm, k-means++
#    seeding, the elbow, and the silhouette score. If somebody handed you an
#    unlabelled dataset tomorrow with no way to check a "real" answer like
#    the iris species, which of these four would you trust least, and why?
answer_5 = """
"""

---
*k-Means: Finding Groups Without Labels · Bluevision-ai Academy*